In [7]:
# Set the CRAN mirror
options(repos = c(CRAN = "https://cran.r-project.org"))

# If not already downloaded in environment, then download DESeq2
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}
BiocManager::install("DESeq2")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.18 (BiocManager 1.30.22), R 4.3.1 (2023-06-16)

Installing package(s) 'BiocVersion', 'DESeq2'

also installing the dependencies ‘bitops’, ‘formatR’, ‘colorspace’, ‘RCurl’, ‘GenomeInfoDbData’, ‘zlibbioc’, ‘abind’, ‘SparseArray’, ‘lambda.r’, ‘futile.options’, ‘nlme’, ‘farver’, ‘labeling’, ‘munsell’, ‘RColorBrewer’, ‘viridisLite’, ‘magrittr’, ‘pkgconfig’, ‘GenomeInfoDb’, ‘XVector’, ‘S4Arrays’, ‘DelayedArray’, ‘futile.logger’, ‘snow’, ‘codetools’, ‘BH’, ‘cpp11’, ‘gtable’, ‘isoband’, ‘MASS’, ‘mgcv’, ‘scales’, ‘tibble’, ‘S4Vectors’, ‘IRanges’, ‘GenomicRanges’, ‘SummarizedExperiment’, ‘BiocGenerics’, ‘Biobase’, ‘BiocParallel’, ‘matrixStats’, ‘locfit’, ‘ggplot2’, ‘MatrixGenerics’, ‘RcppArmadillo’


Old packages: 'cli', 'fansi', 'htmltools', 'jsonlite', 'Matrix', 'rlang',
  'vctrs

In [16]:
install.packages("anndata")

In [1]:
# Load libraries
library("anndata")
library("DESeq2")
library("reticulate")

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

In [5]:
# Use the path to your Python executable in the virtual environment
use_python("/lila/home/forsythb/.virtualenvs/r-reticulate/bin/")

In [6]:
# Look at where Python is located
py_config()

python:         /lila/home/forsythb/.virtualenvs/r-reticulate/bin/python
libpython:      /home/forsythb/anaconda3/lib/libpython3.8.so
pythonhome:     /lila/home/forsythb/.virtualenvs/r-reticulate:/lila/home/forsythb/.virtualenvs/r-reticulate
version:        3.8.18 (default, Sep 11 2023, 13:40:15)  [GCC 11.2.0]
numpy:          /lila/home/forsythb/.virtualenvs/r-reticulate/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by use_python() function

In [8]:
# Import scanpy
sc <- import("scanpy")

# FOR DESeq Metacell Analysis 

## design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression

In [9]:
# Read in the adata
adata <- sc$read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/metacells_noscvi/adata.post.combined.h5ad")

In [10]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [11]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [12]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

converting counts to integer mode



In [13]:
# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names

In [14]:
# Set the reference levels
dds_1$Tumor_Site<-relevel(dds_1$Tumor_Site,ref="Primary")
dds_1$Culture_Media<-relevel(dds_1$Culture_Media,ref="BASE")
dds_1$ZFP_Expression<-relevel(dds_1$ZFP_Expression,ref="CTRL")
dds_1$Patient<-relevel(dds_1$Patient,ref="125")

In [15]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

converting counts to integer mode



In [16]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

-- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

final dispersion estimates

fitting model and testing



In [17]:
# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

[1] "Intercept"                             
[2] "Tumor_Site_Primary_vs_Metastatic"      
[3] "Culture_Media_Dedifferentiated_vs_BASE"
[4] "Culture_Media_HISC_vs_BASE"            
[5] "Patient_146_vs_125"                    
[6] "ZFP_Expression_ZFPKD_vs_CTRL"

log2 fold change (MLE): ZFP Expression ZFPKD vs CTRL 
Wald test p-value: ZFP Expression ZFPKD vs CTRL 
DataFrame with 32485 rows and 6 columns
          baseMean log2FoldChange     lfcSE       stat      pvalue        padj
         <numeric>      <numeric> <numeric>  <numeric>   <numeric>   <numeric>
A1BG       1.02231     0.00211305 0.0479774  0.0440427    0.964870          NA
A1BG-AS1   1.01236     0.00216575 0.0482165  0.0449172    0.964173          NA
A1CF       3.33497    -0.01149037 0.0266570 -0.4310456    0.666435    0.906265
A2M        1.02463    -0.03585165 0.0477423 -0.7509409    0.452688          NA
A2M-AS1    1.02563    -0.03043614 0.0477451 -0.6374714    0.523818          NA
...            ...            ...       ...        ...         ...         ...
ZXDC       3.80565     -0.0227616 0.0247967  -0.917929 3.58656e-01 7.20065e-01
ZYG11A     1.04943     -0.0187291 0.0472598  -0.396300 6.91884e-01          NA
ZYG11B     3.71665     -0.0163685 0.0251284  -0.651394 5.14792e-01 

In [18]:
# Specify the file path where you want to save the results
result_file <- "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/metacell_zfpexp_results.csv"

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/metacell_zfpexp_dds.rds")

# Save the DESeq results
saveRDS(results(dds_1), file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/metacell_zfpexp_results.rds")

Metacells. ZFP Expression. No interaction term.

In [9]:
# Read in the adata
adata <- sc$read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/metacells_noscvi/adata.post.combined.h5ad")

In [10]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [11]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [12]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

converting counts to integer mode



In [13]:
# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names

In [14]:
# Set the reference levels
dds_1$Tumor_Site<-relevel(dds_1$Tumor_Site,ref="Primary")
dds_1$Culture_Media<-relevel(dds_1$Culture_Media,ref="BASE")
dds_1$ZFP_Expression<-relevel(dds_1$ZFP_Expression,ref="CTRL")
dds_1$Patient<-relevel(dds_1$Patient,ref="125")

In [15]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

converting counts to integer mode



In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

estimating size factors

estimating dispersions

gene-wise dispersion estimates



In [ ]:
# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

In [ ]:
# Specify the file path where you want to save the results
result_file <- "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/metacell_zfpexp_results.csv"

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/metacell_zfpexp_dds.rds")

# Save the DESeq results
saveRDS(results(dds_1), file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/metacell_zfpexp_results.rds")

In [ ]:
source /lila/home/forsythb/.virtualenvs/r-reticulate/

In [6]:
# Read in the count matrix 
count_matrix <- read.csv("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/combined/out_post_hvg/metacells_count_matrix.csv")

In [10]:
# Transpose and round count matrix
count_matrix <- t(count_matrix)
count_matrix <- as.matrix(count_matrix)

In [ ]:
# Add a pseudo-count of 1 to all counts
count_matrix <- count_matrix + 1

In [11]:
# Look at the count matrix
count_matrix

X,146_P_BASE_ZFPKD_2_SEACell-139,146_P_BASE_ZFPKD_1_SEACell-90,146_P_HISC_ZFPKD_2_SEACell-82,125_P_BASE_ZFPKD_2_SEACell-29,146_M_Dediff_ZFPKD_2_SEACell-109,146_P_HISC_ZFPKD_1_SEACell-63,125_P_HISC_ZFPKD_2_SEACell-75,146_P_HISC_CTRL_1_SEACell-133,146_M_Dediff_CTRL_1_SEACell-105,146_M_HISC_ZFPKD_1_SEACell-3,⋯,146_P_BASE_ZFPKD_2_SEACell-63,146_P_BASE_CTRL_1_SEACell-204,125_P_HISC_CTRL_1_SEACell-189,146_M_Dediff_CTRL_1_SEACell-33,125_P_HISC_ZFPKD_2_SEACell-116,146_P_BASE_ZFPKD_2_SEACell-56,125_P_BASE_CTRL_1_SEACell-31,146_P_BASE_CTRL_1_SEACell-88,125_P_HISC_ZFPKD_1_SEACell-130,125_P_HISC_CTRL_1_SEACell-278
A1BG,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.8256680,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A1BG.AS1,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A1CF,3.9189490,4.3330428,3.7144981,0.0000000,2.2041534,3.1639819,0.8930490,3.2165162,2.7030099,4.0104060,⋯,4.1836377,0.0000000,0.0000000,2.7722136,0.0000000,3.5031505,2.0745653,3.5013558,0.0000000,0.0000000
A2M,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A2M.AS1,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A2ML1,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.7423085,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A2ML1.AS1,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A2ML1.AS2,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A3GALT2,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
A4GALT,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000


In [3]:
# Read in the observations
adata_obs <- read.csv("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/metacells_noscvi/obs.combined.csv", sep = "\t")

In [19]:
# Look at the observation data
adata_obs

X,Patient,Tumor_Site,Culture_Media,ZFP_Expression,Sample_Name,Batch
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<int>
146_P_BASE_ZFPKD_2_SEACell-139,146,Primary,BASE,ZFPKD,146_P_BASE_ZFPKD_2,5
146_P_BASE_ZFPKD_1_SEACell-90,146,Primary,BASE,ZFPKD,146_P_BASE_ZFPKD_1,5
146_P_HISC_ZFPKD_2_SEACell-82,146,Primary,HISC,ZFPKD,146_P_HISC_ZFPKD_2,6
125_P_BASE_ZFPKD_2_SEACell-29,125,Primary,BASE,ZFPKD,125_P_BASE_ZFPKD_2,0
146_M_Dediff_ZFPKD_2_SEACell-109,146,Metastatic,Dedifferentiated,ZFPKD,146_M_Dediff_ZFPKD_2,4
146_P_HISC_ZFPKD_1_SEACell-63,146,Primary,HISC,ZFPKD,146_P_HISC_ZFPKD_1,6
125_P_HISC_ZFPKD_2_SEACell-75,125,Primary,HISC,ZFPKD,125_P_HISC_ZFPKD_2,1
146_P_HISC_CTRL_1_SEACell-133,146,Primary,HISC,CTRL,146_P_HISC_CTRL_1,6
146_M_Dediff_CTRL_1_SEACell-105,146,Metastatic,Dedifferentiated,CTRL,146_M_Dediff_CTRL_1,4


In [4]:
# Define the column data
coldata <- adata_obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [28]:
coldata

Tumor_Site,Culture_Media,ZFP_Expression,Batch,Patient
<fct>,<fct>,<fct>,<fct>,<fct>
Primary,BASE,ZFPKD,5,146
Primary,BASE,ZFPKD,5,146
Primary,HISC,ZFPKD,6,146
Primary,BASE,ZFPKD,0,125
Metastatic,Dedifferentiated,ZFPKD,4,146
Primary,HISC,ZFPKD,6,146
Primary,HISC,ZFPKD,1,125
Primary,HISC,CTRL,6,146
Metastatic,Dedifferentiated,CTRL,4,146


In [15]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

ERROR: Error in validObject(.Object): invalid class “SummarizedExperiment” object: 
    nb of cols in 'assay' (1) must equal nb of rows in 'colData' (3934)


In [23]:
n_cols_count_matrix <- ncol(count_matrix)
n_rows_coldata <- nrow(coldata)
n_cols_count_matrix
n_rows_coldata

NULL

[1] 3934

In [9]:
count_matrix <- as.numeric(count_matrix)


Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”


In [10]:
count_matrix[is.na(count_matrix)] <- 0


In [24]:
count_matrix

[1] 0.0000000 0.0000000 0.0000000 3.9189490 0.0000000 0.0000000 0.0000000
    [8] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 2.2088860 3.3650782
   [15] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.5468200 2.5551500
   [22] 1.9122210 3.5085434 2.8352743 0.0000000 1.7911654 0.0000000 1.7278738
   [29] 1.7238803 2.4608328 2.8550525 2.2983221 0.0000000 0.0000000 2.7348941
   [36] 1.3177506 0.0000000 1.6161941 1.2897533 0.5202484 0.0000000 0.0000000
   [43] 0.0000000 0.0000000 0.0000000 0.8710784 0.0000000 0.0000000 0.0000000
   [50] 0.5903806 0.0000000 4.1736050 1.2646054 0.0000000 0.0000000 0.0000000
   [57] 1.2440402 3.3879601 0.5903806 0.7349010 3.0839222 0.6502195 0.0000000
   [64] 0.0000000 0.0000000 3.5249265 3.0606642 0.4013477 0.0000000 0.5060770
   [71] 0.0000000 0.0000000 0.4911091 0.0000000 3.4177450 1.8550140 3.4026423
   [78] 3.9282412 1.5578398 0.9395450 0.0000000 0.0000000 0.0000000 0.0000000
   [85] 0.0000000 0.0000000 2.7961010 1.4972641 2.1501709 1.9740594 0.7248106
   [92] 1.4566252 2.3123743 0.9472861 0.9999234 2.7569339 0.0000000 2.3780605
   [99] 3.0569211 3.6863280 2.2511633 2.4743567 2.7455170 1.9439339 0.8701505
  [106] 0.9880736 1.1647162 3.4248902 2.8670016 0.0000000 0.0000000 1.5111898
  [113] 3.6911776 3.0998871 3.8915180 0.0000000 0.0000000 2.7510612 2.3797084
  [120] 0.0000000 4.0026072 1.9724794 2.3626785 2.4428711 0.0000000 1.0165121
  [127] 0.0000000 0.0000000 0.0000000 0.5060770 0.4734145 0.5311426 0.0000000
  [134] 0.2925713 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [141] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [148] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [155] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [162] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [169] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [176] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [183] 0.0000000 0.0000000 0.0000000 0.0000000 1.3328917 0.0000000 0.0000000
  [190] 0.0000000 0.0000000 1.4660858 0.0000000 0.0000000 0.4013477 0.5903806
  [197] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [204] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [211] 0.0000000 0.5202484 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [218] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [225] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [232] 0.0000000 0.0000000 0.0000000 0.5640964 0.0000000 0.0000000 0.0000000
  [239] 0.0000000 0.0000000 0.0000000 0.5094471 0.0000000 0.0000000 0.0000000
  [246] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [253] 1.9803119 0.0000000 0.0000000 1.7584428 0.0000000 0.0000000 0.0000000
  [260] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [267] 0.9124953 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [274] 0.0000000 0.0000000 0.0000000 0.7965751 0.0000000 0.0000000 0.0000000
  [281] 1.2665455 0.5468200 0.7619358 0.0000000 0.0000000 0.0000000 0.0000000
  [288] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.9115346 0.0000000
  [295] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [302] 0.0000000 0.0000000 0.0000000 0.2925713 0.0000000 0.0000000 0.0000000
  [309] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [316] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.4911091 0.0000000
  [323] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [330] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [337] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [344] 0.5094471 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000
  [351] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.5311426 0.0000000
  [358] 0.0000000 0.000000

In [8]:
# Read h5ad file
adata <- read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/metacells_noscvi/adata.post.combined.h5ad")

ERROR: Above error raised while reading key '/layers' of type <class 'h5py._hl.group.Group'> from /.

In [29]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [30]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [ ]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 14.3 GiB”
converting counts to integer mode



In [ ]:
# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names

In [ ]:
# Set the reference levels
dds_1$Tumor_Site<-relevel(dds_1$Tumor_Site,ref="Primary")
dds_1$Culture_Media<-relevel(dds_1$Culture_Media,ref="BASE")
dds_1$ZFP_Expression<-relevel(dds_1$ZFP_Expression,ref="CTRL")
dds_1$Patient<-relevel(dds_1$Patient,ref="125")

In [ ]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

In [21]:
dds_1

class: DESeqDataSet 
dim: 32485 3934 
metadata(1): version
assays(5): counts mu H cooks originalCounts
rownames(32485): A1BG A1BG-AS1 ... ZYX ZZEF1
rowData names(39): baseMean baseVar ... maxCooks replace
colnames(3934): 146_P_BASE_ZFPKD_2_SEACell-139
  146_P_BASE_ZFPKD_1_SEACell-90 ... 125_P_HISC_ZFPKD_1_SEACell-130
  125_P_HISC_CTRL_1_SEACell-278
colData names(7): Tumor_Site Culture_Media ... sizeFactor replaceable

In [18]:
# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

[1] "Intercept"                             
[2] "Tumor_Site_Primary_vs_Metastatic"      
[3] "Culture_Media_Dedifferentiated_vs_BASE"
[4] "Culture_Media_HISC_vs_BASE"            
[5] "Patient_146_vs_125"                    
[6] "ZFP_Expression_ZFPKD_vs_CTRL"

log2 fold change (MLE): ZFP Expression ZFPKD vs CTRL 
Wald test p-value: ZFP Expression ZFPKD vs CTRL 
DataFrame with 32485 rows and 6 columns
          baseMean log2FoldChange     lfcSE       stat      pvalue        padj
         <numeric>      <numeric> <numeric>  <numeric>   <numeric>   <numeric>
A1BG       1.02231     0.00211305 0.0479774  0.0440427    0.964870          NA
A1BG-AS1   1.01236     0.00216575 0.0482165  0.0449172    0.964173          NA
A1CF       3.33497    -0.01149037 0.0266570 -0.4310456    0.666435    0.906265
A2M        1.02463    -0.03585165 0.0477423 -0.7509409    0.452688          NA
A2M-AS1    1.02563    -0.03043614 0.0477451 -0.6374714    0.523818          NA
...            ...            ...       ...        ...         ...         ...
ZXDC       3.80565     -0.0227616 0.0247967  -0.917929 3.58656e-01 7.20065e-01
ZYG11A     1.04943     -0.0187291 0.0472598  -0.396300 6.91884e-01          NA
ZYG11B     3.71665     -0.0163685 0.0251284  -0.651394 5.14792e-01 

In [19]:
# Specify the file path where you want to save the results
result_file <- "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.csv"

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_dds.rds")

# Save the DESeq results
saveRDS(results(dds_1), file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.rds")

In [20]:
# Assuming 'condition' is the factor variable
levels(dds_1$Tumor_Site)

[1] "Metastatic" "Primary"

In [23]:
# Define significant genes based on the value of padj and log2FoldChange
res_1$Significant = res_1$padj < 0.05 & abs(res_1$log2FoldChange) > log2(1.5)

# Filter out non-significant genes
significant_genes_df_1 = res_1[res_1$Significant, ]
significant_genes_df_1 <- significant_genes_df_1[complete.cases(significant_genes_df_1$padj), ]
significant_genes_df_1 <- na.omit(significant_genes_df_1)
significant_genes_df_1

log2 fold change (MLE): ZFP Expression ZFPKD vs CTRL 
Wald test p-value: ZFP Expression ZFPKD vs CTRL 
DataFrame with 6 rows and 7 columns
           baseMean log2FoldChange     lfcSE      stat       pvalue
          <numeric>      <numeric> <numeric> <numeric>    <numeric>
BEX5        1.64228       0.631837 0.0411797   15.3434  3.92197e-53
CADM2       1.29225      -0.649848 0.0408105  -15.9236  4.34826e-57
DANT2       1.66196       0.609677 0.0407152   14.9742  1.08267e-50
H19         3.05664       0.743351 0.0307511   24.1732 4.26124e-129
MEIOB       1.33242      -0.799260 0.0400116  -19.9757  8.95878e-89
MIR548XHG   1.82762       0.821233 0.0401998   20.4288  9.27420e-93
                  padj Significant
             <numeric>   <logical>
BEX5       8.20673e-50        TRUE
CADM2      1.03986e-53        TRUE
DANT2      1.76669e-47        TRUE
H19       7.13332e-125        TRUE
MEIOB      4.99900e-85        TRUE
MIR548XHG  7.76250e-89        TRUE

In [24]:
res_1['ZFP36L2',]

log2 fold change (MLE): ZFP Expression ZFPKD vs CTRL 
Wald test p-value: ZFP Expression ZFPKD vs CTRL 
DataFrame with 1 row and 7 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue        padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
ZFP36L2   4.49937      -0.285815  0.022302  -12.8157 1.33985e-37 1.31936e-34
        Significant
          <logical>
ZFP36L2       FALSE

In [1]:
# Install the R anndata package
install.packages("anndata")

# Run this only if you do not already have an installation of miniconda
#reticulate::install_miniconda(force = TRUE)
#install.packages("reticulate")

# Install the Python anndata package
#anndata::install_anndata()

In [13]:
#install.packages("reticulate")

In [1]:
#Sys.setenv(PATH = paste("lila/home/forsythb/anaconda3/envs/r_4.3.1/bin", Sys.getenv("PATH"), sep = ":"))

In [3]:
#library(reticulate)
#use_python("lila/home/forsythb/anaconda3/envs/r_4.3.1/bin/python")

In [2]:
# Load libraries
library(DESeq2)
library(anndata)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

In [3]:
# Read h5ad file
adata <- read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/combined/out_post_hvg/adata.combined.hvg_5000.h5ad")

In [4]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [5]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [ ]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression + ZFP_Expression:Culture_Media
)

# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names


In [ ]:
# Set the reference levels
dds_1$Tumor_Site<-relevel(dds_1$Tumor_Site,ref="Primary")
dds_1$Culture_Media<-relevel(dds_1$Culture_Media,ref="BASE")
dds_1$ZFP_Expression<-relevel(dds_1$ZFP_Expression,ref="CTRL")
dds_1$Patient<-relevel(dds_1$Patient,ref="125")


In [ ]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression + ZFP_Expression:Culture_Media
)

In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

In [ ]:
# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

In [ ]:
# Specify the file path where you want to save the results
result_file <- "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.csv"

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_dds.rds")

# Save the DESeq results
saveRDS(results(dds_1), file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.rds")